# 逻辑回归实际应用（scikit-learn）

本 Notebook 使用 **scikit-learn 乳腺癌数据集** 演示一个完整的二分类项目流程：
1. 导入依赖、配置中文字体与可视化风格
2. 读取数据并完成基础探索
3. 划分训练/测试集并做标准化
4. 训练基线模型（Logistic Regression）
5. 使用 GridSearchCV 调参
6. 输出评估指标与混淆矩阵
7. 绘制 ROC / PR 曲线并做阈值敏感性分析
8. 分析模型系数（可解释性）
9. 给出单样本推理示例并保存模型工件

说明：
- 标签被重编码为：`1=恶性（高风险）`，`0=良性（低风险）`
- 所有核心步骤都配有详细中文注释，方便学习与复用


In [1]:
# 1. 导入依赖库 + 字体配置 + 绘图风格设置

# 数值计算与表格处理
import numpy as np
import pandas as pd

# 绘图库
import matplotlib.pyplot as plt
from matplotlib import font_manager

# seaborn 是可选库：如果本机没有安装，不影响主流程
# 后续涉及统计图时会自动回退到 matplotlib 实现
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
    print("未检测到 seaborn，将使用 matplotlib 完成对应图表。")

# scikit-learn 相关模块
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
)

# 模型持久化
from joblib import dump


def setup_chinese_font():
    """配置中文字体，避免图中的中文出现方块或乱码。"""

    # 按优先级给出常见中文字体候选
    candidate_fonts = [
        "Microsoft YaHei",   # Windows 常见：微软雅黑
        "SimHei",            # 黑体
        "SimSun",            # 宋体
        "Noto Sans CJK SC",  # 跨平台常见
        "WenQuanYi Micro Hei",
        "Arial Unicode MS",
    ]

    # 当前环境可用字体列表
    available_fonts = {f.name for f in font_manager.fontManager.ttflist}

    for font_name in candidate_fonts:
        if font_name in available_fonts:
            # 设置字体并修复坐标轴负号显示
            plt.rcParams["font.sans-serif"] = [font_name] + plt.rcParams.get("font.sans-serif", [])
            plt.rcParams["axes.unicode_minus"] = False
            print(f"已启用中文字体: {font_name}")
            return font_name

    # 找不到中文字体时，仍保证负号可显示，并给出提示
    plt.rcParams["axes.unicode_minus"] = False
    print("未找到常见中文字体，中文可能显示异常。建议安装微软雅黑或黑体。")
    return None



# 统一绘图风格（网格背景更利于比较曲线）

print("依赖加载完成。")

plt.style.use("seaborn-v0_8-whitegrid")
setup_chinese_font()


未检测到 seaborn，将使用 matplotlib 完成对应图表。
依赖加载完成。
已启用中文字体: Microsoft YaHei


'Microsoft YaHei'

In [2]:
# 2. 读取乳腺癌数据集 + 标签语义重编码 + 基础数据检查

# as_frame=True 会直接返回 pandas DataFrame / Series，便于分析和展示
cancer = load_breast_cancer(as_frame=True)

# 原始特征（共 30 个连续变量）
X = cancer.data.copy()

# 原始标签说明：0=malignant(恶性), 1=benign(良性)
# 为了贴近“风险识别”业务语义，这里重编码为：1=恶性(高风险), 0=良性(低风险)
y = (cancer.target == 0).astype(int)

# 可读标签（仅用于展示，不参与训练）
label_map = {0: "良性", 1: "恶性"}
y_name = y.map(label_map)

# 数据概览
print("数据集名称:", cancer.DESCR.splitlines()[0])
print("特征维度 X:", X.shape)
print("标签维度 y:", y.shape)
print("\n前 5 行特征:")
print(X.head())

# 缺失值检查：医疗数据里这一步非常关键
missing_count = X.isna().sum().sum()
print("\n特征缺失值总数:", missing_count)

# 类别分布检查：用于判断是否需要处理类不平衡
print("\n标签分布(计数):")
print(y_name.value_counts())
print("\n标签分布(占比):")
print((y_name.value_counts(normalize=True) * 100).round(2).astype(str) + "%")


数据集名称: .. _breast_cancer_dataset:
特征维度 X: (569, 30)
标签维度 y: (569,)

前 5 行特征:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal

In [ ]:
# 3. 可视化数据分布（类别分布 + 两个特征散点）

plt.figure(figsize=(12, 4))

# 3.1 类别计数图
plt.subplot(1, 2, 1)
if sns is not None:
    sns.countplot(x=y_name, order=["良性", "恶性"], palette=["#4CAF50", "#F44336"])
else:
    counts = y_name.value_counts().reindex(["良性", "恶性"])
    plt.bar(counts.index, counts.values, color=["#4CAF50", "#F44336"])

plt.title("样本类别分布")
plt.xlabel("类别")
plt.ylabel("样本数")

# 3.2 两个代表性特征的二维散点图
# 通过散点可初步观察两类样本是否存在可分性
plt.subplot(1, 2, 2)
feature_x = "mean radius"
feature_y = "mean texture"

mask_benign = (y == 0)
mask_malignant = (y == 1)

plt.scatter(
    X.loc[mask_benign, feature_x],
    X.loc[mask_benign, feature_y],
    s=25,
    alpha=0.7,
    c="#4CAF50",
    label="良性",
)
plt.scatter(
    X.loc[mask_malignant, feature_x],
    X.loc[mask_malignant, feature_y],
    s=25,
    alpha=0.7,
    c="#F44336",
    label="恶性",
)

plt.xlabel(feature_x)
plt.ylabel(feature_y)
plt.title("特征散点分布（mean radius vs mean texture）")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 4. 划分训练集/测试集 + 特征标准化

# stratify=y: 保持训练集/测试集中的类别比例与总体一致
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 逻辑回归通常对特征尺度敏感：标准化能提升收敛稳定性
scaler = StandardScaler()

# 注意：只在训练集 fit，防止数据泄漏
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("训练集形状:", X_train_scaled.shape)
print("测试集形状:", X_test_scaled.shape)
print("训练集类别计数:", np.bincount(y_train))
print("测试集类别计数:", np.bincount(y_test))


In [ ]:
# 5. 基线模型训练（先得到一个可用版本）

# 参数说明：
# - penalty='l2'：L2 正则化
# - C：正则化强度的倒数，值越小，正则越强
# - class_weight='balanced'：按类别频率自动加权，缓解类别不平衡影响
# - solver='liblinear'：在小中型数据上稳定，且支持 L1/L2
baseline_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    class_weight="balanced",
    max_iter=3000,
    random_state=42,
)

# 训练基线模型
baseline_model.fit(X_train_scaled, y_train)

# 预测标签与概率
y_train_pred_base = baseline_model.predict(X_train_scaled)
y_test_pred_base = baseline_model.predict(X_test_scaled)
y_test_proba_base = baseline_model.predict_proba(X_test_scaled)[:, 1]  # 取“恶性(1)”概率

print("基线模型训练完成。")
print("基线测试集 Accuracy:", f"{accuracy_score(y_test, y_test_pred_base):.4f}")
print("基线测试集 Precision:", f"{precision_score(y_test, y_test_pred_base):.4f}")
print("基线测试集 Recall:", f"{recall_score(y_test, y_test_pred_base):.4f}")
print("基线测试集 F1:", f"{f1_score(y_test, y_test_pred_base):.4f}")
print("基线测试集 ROC-AUC:", f"{roc_auc_score(y_test, y_test_proba_base):.4f}")


In [ ]:
# 6. 网格搜索调参（基于 ROC-AUC 指标）

# 调参思路：
# 1) 在一组候选超参数中搜索最优组合
# 2) 采用分层 K 折交叉验证，保证每折类别比例稳定
# 3) 以 ROC-AUC 作为评分标准，适合医学风险识别场景
param_grid = {
    "C": [0.01, 0.1, 1, 10, 50],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
    "class_weight": ["balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=3000, random_state=42),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid.fit(X_train_scaled, y_train)

# 取最优模型
best_model = grid.best_estimator_

# 在训练集和测试集上预测
y_train_pred = best_model.predict(X_train_scaled)
y_test_pred = best_model.predict(X_test_scaled)
y_train_proba = best_model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print("最优参数:", grid.best_params_)
print("交叉验证最佳 ROC-AUC:", f"{grid.best_score_:.4f}")


In [ ]:
# 7. 统一评估函数：输出核心指标 + 分类报告 + 混淆矩阵

def evaluate_classification(y_true, y_pred, y_proba=None, set_name="测试集"):
    """综合评估二分类模型。"""

    print(f"\n{'=' * 72}")
    print(f"{set_name}评估结果")
    print(f"{'=' * 72}")

    # 1) 基础指标
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    # 2) 概率指标（如果传入概率）
    auc_score = None
    if y_proba is not None:
        auc_score = roc_auc_score(y_true, y_proba)
        print(f"ROC-AUC  : {auc_score:.4f}")

    # 3) 分类报告（按良性/恶性显示）
    print("\n分类报告:")
    print(classification_report(y_true, y_pred, target_names=["良性", "恶性"], digits=4))

    # 4) 混淆矩阵可视化
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    if sns is not None:
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["预测良性", "预测恶性"],
            yticklabels=["真实良性", "真实恶性"],
        )
    else:
        # 回退方案：不用 seaborn 也能画混淆矩阵
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.xticks([0, 1], ["预测良性", "预测恶性"])
        plt.yticks([0, 1], ["真实良性", "真实恶性"])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, cm[i, j], ha="center", va="center", color="black")

    plt.title(f"{set_name}混淆矩阵")
    plt.xlabel("预测标签")
    plt.ylabel("真实标签")
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "auc": auc_score,
    }


# 分别评估训练集与测试集（便于观察是否过拟合）
train_metrics = evaluate_classification(
    y_true=y_train,
    y_pred=y_train_pred,
    y_proba=y_train_proba,
    set_name="训练集",
)

test_metrics = evaluate_classification(
    y_true=y_test,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    set_name="测试集",
)


In [ ]:
# 8. ROC / PR 曲线 + 阈值敏感性分析

# ROC 曲线：观察不同阈值下 TPR 与 FPR 的权衡
fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_proba)
roc_auc_value = auc(fpr, tpr)

# PR 曲线：在正负样本不均衡时更有参考价值
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_test_proba)
pr_auc_value = auc(recall_curve, precision_curve)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color="#1E88E5", label=f"ROC AUC = {roc_auc_value:.4f}")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("假阳性率 (FPR)")
plt.ylabel("真阳性率 (TPR)")
plt.title("测试集 ROC 曲线")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(recall_curve, precision_curve, color="#43A047", label=f"PR AUC = {pr_auc_value:.4f}")
plt.xlabel("召回率 (Recall)")
plt.ylabel("精确率 (Precision)")
plt.title("测试集 PR 曲线")
plt.legend()

plt.tight_layout()
plt.show()

# 阈值敏感性分析：在不同阈值下比较 precision/recall/f1/accuracy
# 医疗场景常常需要根据业务风险调整阈值，而不是固定使用 0.5
threshold_candidates = [0.20, 0.30, 0.50, 0.70, 0.85]
threshold_rows = []

for threshold in threshold_candidates:
    y_pred_t = (y_test_proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
        "f1": f1_score(y_test, y_pred_t),
        "accuracy": accuracy_score(y_test, y_pred_t),
    })

threshold_df = pd.DataFrame(threshold_rows)
print("不同阈值下的指标对比:")
print(threshold_df)


In [ ]:
# 9. 模型可解释性（系数分析）+ 单样本推理 + 模型保存

# 逻辑回归的系数解释：
# - 系数 > 0：特征增大时，更倾向预测为“恶性(1)”
# - 系数 < 0：特征增大时，更倾向预测为“良性(0)”
coef_series = pd.Series(best_model.coef_.ravel(), index=X.columns)

# 取绝对值最大的前 15 个特征，便于聚焦关键变量
top15_features = coef_series.abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(10, 6))
plt.barh(top15_features[::-1], coef_series.loc[top15_features][::-1], color="#FB8C00")
plt.axvline(0, color="black", linewidth=1)
plt.title("逻辑回归系数（Top-15 重要特征）")
plt.xlabel("系数值")
plt.ylabel("特征名")
plt.tight_layout()
plt.show()

print("\n最重要的前10个特征（按绝对值排序）:")
print(coef_series.abs().sort_values(ascending=False).head(10))

# 单样本推理示例：使用测试集第一条样本
sample_index = 0
sample_x = X_test_scaled[sample_index].reshape(1, -1)
sample_prob = best_model.predict_proba(sample_x)[0, 1]
sample_pred = int(sample_prob >= 0.5)

print("\n单样本推理示例:")
print(f"恶性概率: {sample_prob:.4f}")
print(f"预测类别: {sample_pred}（{ '恶性' if sample_pred == 1 else '良性' }）")

# 保存部署所需工件：模型 + 标准化器 + 特征顺序 + 标签映射
artifact = {
    "model": best_model,
    "scaler": scaler,
    "feature_names": list(X.columns),
    "label_map": {0: "良性", 1: "恶性"},
}

dump(artifact, "logistic_breast_cancer_artifact.joblib")
print("\n模型工件已保存为: logistic_breast_cancer_artifact.joblib")
